# 🌡️🤖 Can a Vision-Language Model *read* a thermal image?

### A hands-on tour of **ThermEval** (KDD 2026) — benchmarking VLMs on thermal imagery

**ACM Summer School on AI for Health — Hands-on Notebook**
*Sustainability Lab, IIT Gandhinagar*

---

Modern Vision-Language Models (GPT-4o, Gemini, Qwen-VL, …) are dazzling on
ordinary RGB photos. But almost everything they were trained on is **visible-light**
imagery. Thermal cameras — the same ones behind JoulesEye and ApneaEye in this
school — produce a very different kind of picture: pixel brightness encodes
**temperature**, the palette is **false-colour**, and there is little of the
texture/edge structure VLMs rely on.

So: *do VLMs actually understand thermal images?* **ThermEval** answers this
systematically. This notebook lets you poke at the benchmark yourself:

1. Look at the **7 ThermEval tasks** and load **real benchmark labels** from the
   public repo.
2. Run the **colormap-robustness experiment (Task 2)** on a real thermal frame —
   no model required.
3. *(Optional)* point a **real VLM (Gemini)** at a thermal image and watch it
   answer ThermEval-style questions.
4. **Score** predictions the way ThermEval does (accuracy / MAE).

> Paper: *ThermEval: A Structured Benchmark for Evaluation of Vision-Language
> Models on Thermal Imagery* (KDD 2026, under review).
> Code: github.com/AyushShrivstava/ThermEval_KDD ·
> Data: kaggle.com/datasets/shriayush/thermeval

## 0. The 7 ThermEval tasks

| Task | What the VLM must do | Metric | Data |
|---|---|---|---|
| **T1** Modality ID | Is this image **thermal** or **RGB**? | Accuracy | FLIR, LLVIP |
| **T2** Colormap robustness | Still recognise thermal after recolouring (Gray, Magma, Viridis, Spring, Summer) | Accuracy | FLIR, LLVIP |
| **T3** Human counting | Count the pedestrians | MAE | FLIR, LLVIP |
| **T4** Colorbar interpretation | Find the colourbar, read its temperature range | Accuracy | ThermEval-D |
| **T5** Thermal reasoning | Rank body parts by temperature; who is hotter, left or right? | Accuracy | ThermEval-D |
| **T6** Temperature estimation | Read the °C at a given pixel / region | MAE (°C) | ThermEval-D |
| **T7** Multi-distance temperature | Estimate °C at 2 ft / 6 ft / 10 ft | MAE (°C) | ThermEval-D |

~55,000 thermal VQA pairs; 25 VLMs (0.3 B → 235 B parameters) were evaluated.

## 1. Setup

In [ ]:
import os, io, urllib.request
import numpy as np
import matplotlib.pyplot as plt

try:
    import cv2
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)
    import cv2
from PIL import Image

plt.rcParams.update({"figure.figsize": (11, 3.2), "font.size": 11})
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

def fetch(url, path):
    if not os.path.exists(path) or os.path.getsize(path) < 100:
        urllib.request.urlretrieve(url, path)
    return path

print("Running in Colab:", IN_COLAB)

## 2. A real thermal image to experiment with

ThermEval's images come from FLIR-ADAS, LLVIP and the custom ThermEval-D set
(see the Kaggle link above). To keep this notebook self-contained we instead
grab a **real thermal frame from the lab's own ApneaEye recording** — a genuine
thermal image of a person — and use it as our test image.

In [ ]:
THERMAL_FRAME_URL = "https://raw.githubusercontent.com/AyushShrivstava/ApneaEye_Demo/main/thermal-rec/thermal-rec_2025-11-11_21-04-29.avi"
have_img = False
try:
    p = fetch(THERMAL_FRAME_URL, "/tmp/thermal_frame.avi")
    cap = cv2.VideoCapture(p); ok, frame = cap.read(); cap.release()
    if ok:
        # collapse to a single-channel thermal INTENSITY map (temperature proxy)
        therm = frame.mean(axis=2)
        therm = (therm - therm.min()) / (therm.ptp() + 1e-9)   # normalise 0..1
        have_img = True
        print("Real thermal frame:", therm.shape)
except Exception as e:
    print("Could not fetch the real frame (offline?). Using a synthetic thermal blob.")
    print(" ", e)

if not have_img:
    yy, xx = np.mgrid[0:240, 0:320]
    therm = np.exp(-((xx-170)**2/2200 + (yy-110)**2/1500))     # a warm 'face'
    therm += 0.15*np.exp(-((xx-150)**2/120 + (yy-95)**2/120))  # warm 'nose'
    therm = (therm - therm.min()) / (therm.ptp() + 1e-9)

plt.figure(figsize=(4,3)); plt.imshow(therm, cmap="inferno"); plt.title("Real thermal frame (intensity)"); plt.axis("off"); plt.show()

## 3. Load **real ThermEval labels**

These CSVs are the actual ground-truth questions used in the paper. We pull a
few directly from the benchmark repo so you can see the real format.

In [ ]:
import csv
LABELS = {
    "T1 (modality)":   "https://raw.githubusercontent.com/AyushShrivstava/ThermEval_KDD/main/Labels/T1/T1-llvip.csv",
    "T3 (counting)":   "https://raw.githubusercontent.com/AyushShrivstava/ThermEval_KDD/main/Labels/T3/T3-llvip.csv",
    "T5 (body-part ranking)": "https://raw.githubusercontent.com/AyushShrivstava/ThermEval_KDD/main/Labels/T5/single.csv",
    "T6 (temperature degC)":  "https://raw.githubusercontent.com/AyushShrivstava/ThermEval_KDD/main/Labels/T6/coords.csv",
}
labels = {}
for name, url in LABELS.items():
    try:
        path = fetch(url, "/tmp/" + url.rsplit("/",1)[-1])
        rows = list(csv.DictReader(open(path)))
        labels[name] = rows
        print(f"{name:26}: {len(rows):>5} items   columns={list(rows[0].keys())}")
        print("     e.g.", {k: rows[0][k] for k in rows[0]})
    except Exception as e:
        print(f"{name}: could not fetch ({e})")

## 4. Task 2 hands-on: does the **colour palette** fool the model?

A thermal image is really just a 2-D field of temperatures; the colours are a
choice. ThermEval Task 2 re-renders the *same* thermal data in different
colormaps and checks whether a VLM still calls it "thermal". Many models do
**not** — under perceptually-uniform palettes like *Viridis* they start
describing the image as an ordinary RGB photo.

Below we apply the five ThermEval colormaps to our real frame. The underlying
temperatures are identical — only the colours change.

In [ ]:
CMAPS = ["gray", "magma", "viridis", "spring", "summer"]   # the ThermEval-2 set

def colorize(intensity01, cmap):
    rgba = plt.get_cmap(cmap)(intensity01)          # HxWx4 in 0..1
    return (rgba[..., :3] * 255).astype(np.uint8)   # HxWx3 uint8

variants = {c: colorize(therm, c) for c in CMAPS}

fig, ax = plt.subplots(1, len(CMAPS), figsize=(14, 3))
for a, c in zip(ax, CMAPS):
    a.imshow(variants[c]); a.set_title(c); a.axis("off")
fig.suptitle("Same thermal data, five palettes — all are still 'thermal'", y=1.05)
plt.tight_layout(); plt.show()
print("A robust VLM should answer 'thermal' for ALL five. ThermEval shows many flip"
      " to 'RGB/visible' once the palette stops looking like classic fire colours.")

## 5. *(Optional)* Ask a real VLM — Gemini

To actually query a model you need a **free Google AI Studio API key**
(aistudio.google.com → *Get API key*). Paste it below. Without a key this
section is skipped and the notebook still completes.

```python
import os
os.environ["GOOGLE_API_KEY"] = "your-key-here"   # or use Colab Secrets
```

In [ ]:
HAS_KEY = bool(os.environ.get("GOOGLE_API_KEY"))

def ask_gemini(rgb_uint8, question, model="gemini-flash-latest"):
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    img = Image.fromarray(rgb_uint8)
    resp = genai.GenerativeModel(model).generate_content([question, img])
    return resp.text.strip()

if HAS_KEY:
    try:
        import google.generativeai  # noqa
    except ImportError:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-generativeai"], check=True)

    base = colorize(therm, "magma")
    q1 = "Is this image a THERMAL/infrared image or an ordinary RGB photo? Answer one word."
    q3 = "How many people are in this thermal image? Answer with a single number."
    print("T1  thermal-vs-rgb :", ask_gemini(base, q1))
    print("T3  person count   :", ask_gemini(base, q3))
    print("\nT2 robustness across palettes:")
    for c in CMAPS:
        print(f"   {c:8}: {ask_gemini(colorize(therm, c), q1)}")
else:
    print("No GOOGLE_API_KEY set — skipping the live VLM calls.")
    print("Set one (free at aistudio.google.com) and re-run this cell to see a real")
    print("VLM answer ThermEval-style questions on the thermal frame above.")

## 6. Scoring, the ThermEval way

Two metrics cover the benchmark: **accuracy** for the categorical tasks
(T1, T2, T5) and **mean absolute error** for the numeric ones (T3 counting,
T6/T7 temperature). Below we implement both and run them on the **real T1 and
T3 ground truth** using simple, transparent *baseline* predictors — so you can
see the full scoring pipeline even without running a model. Plug a VLM's
predictions in place of the baselines to get its real score.

In [ ]:
def accuracy(preds, gts):
    preds = [str(p).strip().lower() for p in preds]
    gts   = [str(g).strip().lower() for g in gts]
    return np.mean([p == g for p, g in zip(preds, gts)])

def mae(preds, gts):
    return float(np.mean(np.abs(np.asarray(preds, float) - np.asarray(gts, float))))

# T1 — baseline that always answers "Thermal"
if "T1 (modality)" in labels:
    g = [r["ground_truth"] for r in labels["T1 (modality)"]]
    base_pred = ["Thermal"] * len(g)
    acc = accuracy(base_pred, g)
    print(f"T1 accuracy, 'always Thermal' baseline : {acc:.3f}  (n={len(g)})")
    print(f"   -> {acc*100:.0f}%: the T1 set is balanced thermal vs RGB, so 'always Thermal'")
    print("      is no better than a coin flip. A real VLM has to actually look.")

# T3 — baseline that always predicts the mean count
if "T3 (counting)" in labels:
    counts = [float(r["ground_truth"]) for r in labels["T3 (counting)"]]
    mean_pred = [np.mean(counts)] * len(counts)
    print(f"T3 counting MAE, 'predict-the-mean' baseline : {mae(mean_pred, counts):.2f} people  (n={len(counts)})")
    print(f"   ground-truth counts range {int(min(counts))}..{int(max(counts))}, mean {np.mean(counts):.1f}")

## 7. What ThermEval found (and why it matters for health sensing)

Across 25 VLMs, the paper documents **systematic failure modes** on thermal
imagery, e.g.:

- **Palette dependence (T2):** accuracy at telling "is this thermal?" swings with
  the colormap — models lean on *fire-coloured* cues rather than understanding
  the modality.
- **Weak counting (T3)** and **poor absolute temperature reading (T6/T7):**
  reading °C off a colourbar is far from solved.
- **Reasoning gaps (T5):** ranking body parts by temperature, or judging which
  person is warmer, remains hard.

Why care here? JoulesEye, ApneaEye and SpiroMask all turn raw sensor signals
into health markers with **purpose-built** pipelines. ThermEval is the
reality-check on the tempting shortcut — *"just ask a big VLM"* — and shows why,
for thermal health imaging today, careful task-specific modelling still wins.

Run the failure-mode figure from the paper:

In [ ]:
try:
    p = fetch("https://raw.githubusercontent.com/AyushShrivstava/ThermEval_KDD/main/images/ThermEval_Failures.jpg", "/tmp/thermeval_failures.jpg")
    img = plt.imread(p)
    plt.figure(figsize=(11, 5)); plt.imshow(img); plt.axis("off")
    plt.title("ThermEval: documented VLM failure modes on thermal imagery"); plt.show()
except Exception as e:
    print("Could not fetch the figure (offline?).", e)

## 8. Limitations & things to try

- This is a **teaser** of ThermEval. The full benchmark (Kaggle dataset) has
  ~55k pairs across FLIR-ADAS, LLVIP and ThermEval-D with temperature matrices
  (`.tiff`) for the °C tasks.
- The live demo uses one Gemini model; ThermEval evaluates 25 VLMs.

**Exercises**
1. Add your `GOOGLE_API_KEY` and measure **T2 accuracy** across the five
   colormaps on several frames — does your model flip on *Viridis*?
2. Swap in another provider (OpenAI, Anthropic, Qwen-VL via OpenRouter) behind
   the same `ask_*` interface and compare.
3. Download a few **FLIR/LLVIP** images (links in the repo) and run real **T3
   counting**; compute MAE against the provided labels.
4. For **T6**, load a ThermEval-D temperature matrix and ask the VLM to read the
   °C at a marked pixel — then score with `mae()`.

*Educational benchmark demo. Not a medical device.*

---
*Made for the ACM Summer School on AI for Health · ThermEval, KDD 2026.*